<a href="https://colab.research.google.com/github/ERA-Software/computational-data-analysis/blob/main/notebooks/S4_PCA_UMAP_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA and UMAP Demo
## Lecture 4: Exploratory Data Analysis and Visualization

This notebook accompanies the live demo (30 min). We apply PCA and UMAP to two datasets:

1. **Palmer Penguins** (344 obs., 4 numeric features) for interpretable, small-scale examples.
2. **Fashion-MNIST** (10,000 images, 784 pixels) for a high-dimensional case where the methods diverge.

After the demo, use the "Try this yourself" cells at the end to experiment.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# UMAP is a separate install: pip install umap-learn
import umap

# For Fashion-MNIST
from sklearn.datasets import fetch_openml

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

SEED = 42
np.random.seed(SEED)

---
## Segment 1: PCA on Palmer Penguins (10 min)

### Load and inspect

In [ ]:
# palmerpenguins is available via: pip install palmerpenguins
# Alternatively, load from CSV or seaborn.
try:
    from palmerpenguins import load_penguins
    penguins_raw = load_penguins()
except ImportError:
    # Fallback: load from seaborn
    import seaborn as sns
    penguins_raw = sns.load_dataset("penguins")

print(f"Shape: {penguins_raw.shape}")
penguins_raw.head()

In [ ]:
# Drop rows with missing numeric values; keep the species label.
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = penguins_raw.dropna(subset=numeric_cols).copy()
X_pen = penguins[numeric_cols].values
species = penguins["species"].values

print(f"After dropping NAs: {X_pen.shape[0]} observations, {X_pen.shape[1]} features")
print(f"\nDescriptive statistics (note the scale differences):")
penguins[numeric_cols].describe().round(1)

**Before proceeding:** Look at the scales. `body_mass_g` ranges from ~2700 to ~6300. `bill_depth_mm` ranges from ~13 to ~22. 
What will happen if we run PCA without standardizing?

### Standardize and fit PCA

In [ ]:
scaler = StandardScaler()
X_pen_std = scaler.fit_transform(X_pen)

pca_pen = PCA(n_components=4)
Z_pen = pca_pen.fit_transform(X_pen_std)

# Variance explained
var_explained = pca_pen.explained_variance_ratio_
cumvar = np.cumsum(var_explained)

print("Variance explained per component:")
for i, (v, c) in enumerate(zip(var_explained, cumvar)):
    print(f"  PC{i+1}: {v:.3f}  (cumulative: {c:.3f})")

**Question:** How many components would you keep? 
(2 components capture ~69% individually and ~88% cumulatively. A clear elbow after PC2.)

In [ ]:
# Scree plot
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(range(1, 5), var_explained, color="#1e5aa8", alpha=0.7, label="Individual")
ax.plot(range(1, 5), cumvar, "o-", color="#e04040", label="Cumulative")
ax.set_xlabel("Principal component")
ax.set_ylabel("Fraction of variance explained")
ax.set_xticks(range(1, 5))
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title("Scree plot (Palmer Penguins, standardized)")
plt.tight_layout()
plt.show()

### 2D scatter colored by species

In [ ]:
species_colors = {"Adelie": "#ff7f0e", "Chinstrap": "#2ca02c", "Gentoo": "#1f77b4"}

fig, ax = plt.subplots(figsize=(7, 5))
for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    mask = species == sp
    ax.scatter(Z_pen[mask, 0], Z_pen[mask, 1], label=sp,
               c=species_colors[sp], alpha=0.6, edgecolors="white", linewidths=0.3, s=40)
ax.set_xlabel(f"PC1 ({var_explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.1%} variance)")
ax.set_title("PCA on Palmer Penguins (standardized)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Question:** Gentoo separates cleanly on PC1. What does PC1 represent?
Let's look at the loadings.

In [ ]:
# Loadings
loadings = pd.DataFrame(
    pca_pen.components_.T,
    index=numeric_cols,
    columns=[f"PC{i+1}" for i in range(4)],
)
print("PCA loadings (rows = original features, columns = components):")
print(loadings.round(3))

**Interpretation:** PC1 has roughly equal positive weights on `bill_length`, `flipper_length`, and `body_mass`, with a negative weight on `bill_depth`. 
This is a "body size with bill shape" axis that separates Gentoo (large body, long bill, shallow depth) from Adelie and Chinstrap.

### Pitfall: PCA without standardizing

In [ ]:
# Run PCA on raw (unstandardized) data
pca_raw = PCA(n_components=4)
Z_raw = pca_raw.fit_transform(X_pen)  # no scaler!

print("Loadings WITHOUT standardization:")
loadings_raw = pd.DataFrame(
    pca_raw.components_.T,
    index=numeric_cols,
    columns=[f"PC{i+1}" for i in range(4)],
)
print(loadings_raw.round(3))
print(f"\nVariance explained by PC1: {pca_raw.explained_variance_ratio_[0]:.3f}")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    mask = species == sp
    axes[0].scatter(Z_pen[mask, 0], Z_pen[mask, 1], label=sp,
                    c=species_colors[sp], alpha=0.6, edgecolors="white", linewidths=0.3, s=30)
    axes[1].scatter(Z_raw[mask, 0], Z_raw[mask, 1], label=sp,
                    c=species_colors[sp], alpha=0.6, edgecolors="white", linewidths=0.3, s=30)

axes[0].set_title("PCA (standardized)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(frameon=False, fontsize=9)

axes[1].set_title("PCA (NOT standardized)")
axes[1].set_xlabel("PC1 (dominated by body_mass_g)")
axes[1].set_ylabel("PC2")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

**Key point:** Without standardization, `body_mass_g` (measured in grams) has variance ~10^6 times larger than `bill_depth_mm` (in millimeters). 
PC1 becomes almost entirely `body_mass_g`. The PCA is mathematically correct given the input, but the input was poorly prepared.

---
## Segment 2: UMAP on Palmer Penguins (5 min)

### Default UMAP

In [ ]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED)
Z_umap = reducer.fit_transform(X_pen_std)

# Side-by-side: PCA vs. UMAP
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    mask = species == sp
    axes[0].scatter(Z_pen[mask, 0], Z_pen[mask, 1], label=sp,
                    c=species_colors[sp], alpha=0.6, edgecolors="white", linewidths=0.3, s=30)
    axes[1].scatter(Z_umap[mask, 0], Z_umap[mask, 1], label=sp,
                    c=species_colors[sp], alpha=0.6, edgecolors="white", linewidths=0.3, s=30)

axes[0].set_title("PCA")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(frameon=False, fontsize=9)

axes[1].set_title("UMAP (n_neighbors=15, min_dist=0.1)")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

### Hyperparameter exploration

UMAP output depends on two key hyperparameters:
- `n_neighbors`: how many neighbors define "local" (small = fine detail, large = global structure).
- `min_dist`: how tightly clusters are packed (small = tight, large = spread out).

**Question:** Which setting makes clusters tightest? Which preserves more of the continuous within-species structure?

In [ ]:
n_neighbors_vals = [5, 50]
min_dist_vals = [0.01, 0.5]

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for i, nn in enumerate(n_neighbors_vals):
    for j, md in enumerate(min_dist_vals):
        reducer_ij = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=SEED)
        Z_ij = reducer_ij.fit_transform(X_pen_std)
        ax = axes[i, j]
        for sp in ["Adelie", "Chinstrap", "Gentoo"]:
            mask = species == sp
            ax.scatter(Z_ij[mask, 0], Z_ij[mask, 1], label=sp,
                       c=species_colors[sp], alpha=0.6, edgecolors="white",
                       linewidths=0.3, s=25)
        ax.set_title(f"n_neighbors={nn}, min_dist={md}", fontsize=10)
        ax.set_xlabel("UMAP 1", fontsize=9)
        ax.set_ylabel("UMAP 2", fontsize=9)
        if i == 0 and j == 0:
            ax.legend(frameon=False, fontsize=8)

plt.suptitle("UMAP hyperparameter grid (Palmer Penguins)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Key point:** There is no single "correct" UMAP embedding. The output changes with hyperparameters. 
Small `n_neighbors` + small `min_dist` produces the tightest clusters but may fragment continuous structure. 
Large `n_neighbors` smooths out local detail.

---
## Segment 3: Scaling Up with Fashion-MNIST (10 min)

Fashion-MNIST has 70,000 grayscale 28x28 images in 10 classes. We use a 10,000-sample subset for speed.

In [ ]:
# Load Fashion-MNIST (first load may take a minute)
fmnist = fetch_openml("Fashion-MNIST", version=1, as_frame=False, parser="auto")
X_fmnist_full = fmnist.data.astype(np.float32)
y_fmnist_full = fmnist.target.astype(int)

# Subsample 10,000 points for the demo
rng = np.random.RandomState(SEED)
idx = rng.choice(len(X_fmnist_full), size=10_000, replace=False)
X_fm = X_fmnist_full[idx]
y_fm = y_fmnist_full[idx]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

print(f"Shape: {X_fm.shape}  (10,000 images, 784 pixels each)")
print(f"Classes: {class_names}")

In [ ]:
# Show a few example images
fig, axes = plt.subplots(2, 10, figsize=(12, 3))
for c in range(10):
    examples = X_fm[y_fm == c][:2]
    for row in range(2):
        axes[row, c].imshow(examples[row].reshape(28, 28), cmap="gray_r")
        axes[row, c].axis("off")
    axes[0, c].set_title(class_names[c], fontsize=7)
plt.suptitle("Fashion-MNIST samples (2 per class)", fontsize=11)
plt.tight_layout()
plt.show()

### PCA on Fashion-MNIST

In [ ]:
# Standardize pixel values (zero mean, unit variance per pixel)
scaler_fm = StandardScaler()
X_fm_std = scaler_fm.fit_transform(X_fm)

# Fit PCA with many components to see the scree plot
pca_fm = PCA(n_components=50)
Z_fm_pca = pca_fm.fit_transform(X_fm_std)

cumvar_fm = np.cumsum(pca_fm.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, 51), cumvar_fm, "o-", markersize=3, color="#1e5aa8")
ax.axhline(0.9, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("Scree plot (Fashion-MNIST, 10K subset)")
ax.text(35, 0.91, "90% threshold", fontsize=9, color="gray")
plt.tight_layout()
plt.show()

# How many components for 90%?
n90 = np.searchsorted(cumvar_fm, 0.9) + 1
print(f"Components needed for 90% variance: {n90}")
print(f"Variance in first 2 components: {cumvar_fm[1]:.3f}")

**Key observation:** No sharp elbow. The variance is spread across many
components (typical for image data). The first 2 components capture only
~25% of total variance.

In [ ]:
# PCA 2D scatter colored by class
fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.cm.tab10
for c in range(10):
    mask = y_fm == c
    ax.scatter(Z_fm_pca[mask, 0], Z_fm_pca[mask, 1],
               c=[cmap(c)], alpha=0.3, s=5, label=class_names[c])
ax.set_xlabel(f"PC1 ({pca_fm.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_fm.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA on Fashion-MNIST (first 2 components)")
ax.legend(fontsize=7, frameon=False, markerscale=3, ncol=2)
plt.tight_layout()
plt.show()

**Question:** Classes overlap heavily. Does this mean PCA "failed"?

No. PCA finds directions of maximum variance. The highest-variance directions in images correspond to overall brightness and contrast, not semantic category.
PCA is doing exactly what it is designed to do; the issue is that the variance-maximizing directions are not the class-separating directions.

### UMAP on Fashion-MNIST

In [ ]:
# UMAP (takes ~15-30 seconds on 10K points)
reducer_fm = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED)
Z_fm_umap = reducer_fm.fit_transform(X_fm_std)

fig, ax = plt.subplots(figsize=(8, 6))
for c in range(10):
    mask = y_fm == c
    ax.scatter(Z_fm_umap[mask, 0], Z_fm_umap[mask, 1],
               c=[cmap(c)], alpha=0.3, s=5, label=class_names[c])
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("UMAP on Fashion-MNIST")
ax.legend(fontsize=7, frameon=False, markerscale=3, ncol=2)
plt.tight_layout()
plt.show()

**Question:** Why does UMAP separate classes when PCA does not?

UMAP preserves local neighborhoods: a sneaker is locally near other sneakers in pixel space, and UMAP respects that proximity. 
PCA's linear projection cannot capture the nonlinear manifold on which each class lives.

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for c in range(10):
    mask = y_fm == c
    axes[0].scatter(Z_fm_pca[mask, 0], Z_fm_pca[mask, 1],
                    c=[cmap(c)], alpha=0.3, s=4, label=class_names[c])
    axes[1].scatter(Z_fm_umap[mask, 0], Z_fm_umap[mask, 1],
                    c=[cmap(c)], alpha=0.3, s=4, label=class_names[c])

axes[0].set_title("PCA (2 components)")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].set_title("UMAP (n_neighbors=15, min_dist=0.1)")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")

# Shared legend
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, fontsize=8,
           frameon=False, markerscale=3)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### The danger: UMAP distances are not meaningful

In [ ]:
# Run UMAP twice with different random seeds
Z_run1 = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=1).fit_transform(X_fm_std)
Z_run2 = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=99).fit_transform(X_fm_std)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for c in range(10):
    mask = y_fm == c
    axes[0].scatter(Z_run1[mask, 0], Z_run1[mask, 1],
                    c=[cmap(c)], alpha=0.3, s=4)
    axes[1].scatter(Z_run2[mask, 0], Z_run2[mask, 1],
                    c=[cmap(c)], alpha=0.3, s=4)

axes[0].set_title("UMAP run 1 (seed=1)")
axes[1].set_title("UMAP run 2 (seed=99)")
for ax in axes:
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
plt.suptitle("Same data, same hyperparameters, different seeds", fontsize=12)
plt.tight_layout()
plt.show()

**Key point:** Cluster positions move across runs. The clusters themselves are real (consistent membership), but the distances between clusters are not meaningful. 
You cannot say "cluster A is closer to cluster B than to cluster C" from a UMAP plot. 
PCA distances, by contrast, are meaningful because the projection is linear and deterministic.

In [ ]:
# Extreme hyperparameter: n_neighbors=2 fragments the plot
Z_fragmented = umap.UMAP(n_neighbors=2, min_dist=0.01, random_state=SEED).fit_transform(X_fm_std)

fig, ax = plt.subplots(figsize=(7, 5))
for c in range(10):
    mask = y_fm == c
    ax.scatter(Z_fragmented[mask, 0], Z_fragmented[mask, 1],
               c=[cmap(c)], alpha=0.3, s=4)
ax.set_title("UMAP with n_neighbors=2, min_dist=0.01 (fragmented)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

With `n_neighbors=2`, UMAP only considers each point's 2 nearest neighbors.
The graph becomes disconnected, and the embedding fragments into noise.
This is not a "failure" of UMAP; it is the expected behavior when the neighborhood size is too small to capture any global structure.

---
## Segment 4: Wrap-up (5 min)

### Summary table

| Property | PCA | UMAP |
|---|---|---|
| Linear? | Yes | No |
| Preserves global distances? | Yes | No |
| Preserves local neighborhoods? | Approximately | Yes |
| Deterministic? | Yes | No |
| Scales to millions? | Yes | Yes |
| Interpretable axes? | Yes (loadings) | No |
| Safe for downstream features? | Yes | No |

### Connection to EDA purposes

| EDA Purpose | PCA serves it by... | UMAP serves it by... |
|---|---|---|
| Describe | Variance structure, scree plot | (not ideal for description) |
| Diagnose | Scale problems (the standardization pitfall) | Duplicate/near-duplicate groups |
| Discover | (limited for nonlinear structure) | Cluster structure, outlier groups |
| Decide | How many components for a downstream model | Whether the embedding separates classes |